# Wisconsin Diagnostic Breast Cancer: quick exploration

This notebook is for exploratory data analysis only. It loads the same built-in scikit-learn dataset used by the training script and recodes malignant tumours as 1. Predictive preprocessing and model evaluation are handled in `src/train.py`.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import seaborn as sns

root = Path.cwd()
if not (root / 'src').exists():
    root = root.parent
sys.path.insert(0, str(root))
from src.data import load_dataset

X, y = load_dataset()
class_names = y.map({0: 'Benign', 1: 'Malignant'})
print(f'Shape: {X.shape}')
X.head()

## Structure and class balance

There are 30 numeric measurements for each sample. The target is stratified when the training script makes its train/test split so both tumour classes remain represented.

In [ ]:
X.info()
print('Missing feature values:', int(X.isna().sum().sum()))
print('Class counts:')
print(class_names.value_counts().reindex(['Benign', 'Malignant']))

In [ ]:
sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x=class_names, order=['Benign', 'Malignant'], ax=axes[0], color='#4c9f70')
axes[0].set(title='Class balance', xlabel='Tumour class', ylabel='Samples')
key = X[['radius_mean', 'area_mean', 'concavity_mean', 'concave points_mean']].copy()
key['Class'] = class_names.values
sns.boxplot(data=key, x='Class', y='radius_mean', order=['Benign', 'Malignant'], ax=axes[1], palette={'Benign': '#4c9f70', 'Malignant': '#cf5c60'})
axes[1].set(title='Mean radius by class', xlabel='Tumour class', ylabel='Mean radius')
fig.tight_layout()

## Correlations between selected features

The measurements are not independent: size features in particular can be strongly correlated. The model pipeline standardizes the inputs and uses `SelectKBest` inside cross-validation, preventing feature selection from seeing the held-out test set.

In [ ]:
subset = ['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave points_mean']
plt.figure(figsize=(9, 7))
sns.heatmap(X[subset].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation among selected mean measurements')
plt.tight_layout()